# 07_decomposition: Step 7 - where do the savings come from? Walls vs setpoints.

```
Runs NSGA-II (best tuned hyperparameters, 10 seeds, 25,000 evaluations) on three
variants of each case study:
  * setpoints only  - walls fixed at the existing layout, zone setpoints 24-27 degC
  * walls only      - setpoints fixed at 24 degC (the policy floor), walls movable
  * walls + setpoints (the full problem)
The walls-only result is independent of the setpoint-energy coefficient BETA,
so it isolates what the movable-wall layout contributes on its own.
Outputs -> outputs/nsga2/decomposition.csv
```

Dikonversi dari `scripts/07_decomposition.py`. Jalankan dari folder `notebooks/`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.layout import T_MIN, case_study_1, case_study_2, evaluate
from obep_mlpaper.nsga import run_nsga2
from obep_mlpaper.surrogate import Surrogate

EPS = 1e-6
sur = Surrogate.load(ROOT / "outputs" / "models" / "surrogate.joblib").single_thread()
out = ROOT / "outputs" / "nsga2"
SEEDS = range(1, 11)

In [2]:
def variant(make, kind):
    sc = make()
    nw = sc.n_walls
    if kind == "setpoints only":
        sc.xl[:nw] = sc.x_init[1:-1]
        sc.xu[:nw] = sc.x_init[1:-1] + EPS
    elif kind == "walls only":
        sc.xl[nw:] = T_MIN
        sc.xu[nw:] = T_MIN + EPS
    return sc

In [3]:
def worst_abs_pmv(r):
    return np.where(r["occ"] > 0, np.abs(r["pmv"]), 0.0).max(axis=1)

In [4]:
rows = []
for make in (case_study_1, case_study_2):
    name = make().name
    summ = json.loads((out / f"{name}_summary.json").read_text())
    hp = summ["best_config_hyperparameters"]
    for kind in ("setpoints only", "walls only", "walls + setpoints"):
        sc = variant(make, kind)
        res = Parallel(n_jobs=2)(delayed(run_nsga2)(sc, sur, hp, s, 25_000) for s in SEEDS)
        f0 = res[0]["f0"]
        F = np.vstack([r["F"] for r in res])
        X = np.vstack([r["X"] for r in res])
        ok = worst_abs_pmv(evaluate(sc, sur, X)) <= 0.5
        rows.append(dict(
            case=name, variant=kind, hv_mean=np.mean([r["hv"] for r in res]),
            max_energy_saving_pct=100 * (1 - F[:, 0].min() / f0[0]),
            max_comfort_gain_pct=100 * (1 - F[:, 1].min() / f0[1]),
            energy_saving_with_all_pmv_ok_pct=(100 * (1 - F[ok, 0].min() / f0[0])) if ok.any() else np.nan))
        print(rows[-1], flush=True)
pd.DataFrame(rows).to_csv(out / "decomposition.csv", index=False)
print(pd.DataFrame(rows).round(2).to_string(index=False))

{'case': 'case1_simple', 'variant': 'setpoints only', 'hv_mean': np.float64(1.0019228770992306), 'max_energy_saving_pct': np.float64(21.78590640129101), 'max_comfort_gain_pct': np.float64(99.8572260657724), 'energy_saving_with_all_pmv_ok_pct': np.float64(17.038847682029424)}


{'case': 'case1_simple', 'variant': 'walls only', 'hv_mean': np.float64(0.25856813251682687), 'max_energy_saving_pct': np.float64(1.2056607603169178), 'max_comfort_gain_pct': np.float64(0.5869400472330089), 'energy_saving_with_all_pmv_ok_pct': nan}


{'case': 'case1_simple', 'variant': 'walls + setpoints', 'hv_mean': np.float64(1.020576518041615), 'max_energy_saving_pct': np.float64(22.76977551271545), 'max_comfort_gain_pct': np.float64(99.88100219385562), 'energy_saving_with_all_pmv_ok_pct': np.float64(17.629782648256544)}


{'case': 'case2_watson', 'variant': 'setpoints only', 'hv_mean': np.float64(1.0456891685002911), 'max_energy_saving_pct': np.float64(21.785906840777002), 'max_comfort_gain_pct': np.float64(99.96023806900023), 'energy_saving_with_all_pmv_ok_pct': np.float64(19.80326524537731)}


{'case': 'case2_watson', 'variant': 'walls only', 'hv_mean': np.float64(0.29203320830806934), 'max_energy_saving_pct': np.float64(6.50807025024176), 'max_comfort_gain_pct': np.float64(2.3510800254426645), 'energy_saving_with_all_pmv_ok_pct': nan}


{'case': 'case2_watson', 'variant': 'walls + setpoints', 'hv_mean': np.float64(1.1192292614410833), 'max_energy_saving_pct': np.float64(27.259193296889784), 'max_comfort_gain_pct': np.float64(99.97594500885207), 'energy_saving_with_all_pmv_ok_pct': np.float64(25.757372045694137)}


        case           variant  hv_mean  max_energy_saving_pct  max_comfort_gain_pct  energy_saving_with_all_pmv_ok_pct
case1_simple    setpoints only     1.00                  21.79                 99.86                              17.04
case1_simple        walls only     0.26                   1.21                  0.59                                NaN
case1_simple walls + setpoints     1.02                  22.77                 99.88                              17.63
case2_watson    setpoints only     1.05                  21.79                 99.96                              19.80
case2_watson        walls only     0.29                   6.51                  2.35                                NaN
case2_watson walls + setpoints     1.12                  27.26                 99.98                              25.76
